# Phase 4 follow-up: fixed comparison on new questions and Flask

Upload **phase4_followup_source.zip** to the top level of My Drive. Select an **A100 GPU**,
run all cells, approve Drive mounting, and keep the session open until the results ZIP downloads.
Use this notebook and ZIP as a matching pair. There is no embedded archive or Base64 cell.

This runs 49 newly authored questions plus 80 frozen-question regressions across Requests,
Express and Flask. The new labels distinguish required from supporting evidence. They are
source-checked by the author; independent label review is still pending. New symbol lookups
can target previously evaluated implementations, so newly authored is not fully independent.

The matrix is fixed: dense, current hybrid, rerank20 and a reranked 20-candidate union reserving
five BM25 slots. Penalty and symbol routing are diagnostics, not selected production policies.
No weights are tuned. It logs full lists/scores, reranker pairs, separate v1 scores, actual warm
p50/p95 timings, cold first rerank and CUDA memory. Diagnostic top50 scores explain movements
for requests-036 and express-028. Repetitions do not increase the number of independent questions.
The primary comparison uses production chunk headers. The original common-header study remains
a separate benchmark, so its scores must not be treated as a direct before/after improvement.

Bring **phase4_followup_results.zip** back for review. This experiment never changes production
defaults or marks Phase 4 complete automatically.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_followup_source.zip"

## Load from Drive

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_followup_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "3d0fa8e954b553543428423fcad326c9c5ccf7437a4bc967fa5cad929db4ffe3"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_followup_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

## Install pinned dependencies with uv

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system', '--python',
                sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

## Run the fixed matrix

In [ ]:
import json, os, pathlib, subprocess, sys, time
REPOS = pathlib.Path('/content/phase4_followup_repos')
RESULTS = pathlib.Path('/content/phase4_followup_results')
REPOS.mkdir(exist_ok=True)
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'REPO_RAG_DATA_DIR': '/content/unused_production_data', 'REPO_RAG_COMPUTE_DEVICE': 'cuda',
       'REPO_RAG_EMBEDDING_DTYPE': 'float32', 'PYTHONUNBUFFERED': '1',
       'TOKENIZERS_PARALLELISM': 'false'}
manifest = json.loads((PAYLOAD / 'eval/datasets/phase4_v2/manifest.json').read_text())
started = time.perf_counter()
for entry in manifest['datasets']:
    repo, sha = entry['repo'], entry['commit_sha']
    checkout = REPOS / repo.replace('/', '__')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                        'https://github.com/' + repo, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', sha], check=True)
    dataset = PAYLOAD / 'eval/datasets/phase4_v2' / entry['file']
    if hashlib.sha256(dataset.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Dataset hash mismatch: ' + entry['file'])
    command = [sys.executable, str(PAYLOAD / 'eval/run_followup_eval.py'), '--repo', repo,
               '--checkout', str(checkout), '--dataset', str(dataset), '--output', str(RESULTS)]
    subprocess.run(command + ['--mode', 'validate'], env=env, check=True)
    subprocess.run(command + ['--mode', 'full'], env=env, check=True)
print(f'Full matrix elapsed: {time.perf_counter() - started:.1f} seconds')
subprocess.run([sys.executable, str(PAYLOAD / 'eval/summarize_followup.py'),
                '--results', str(RESULTS)], env=env, check=True)

## Download results

In [ ]:
from google.colab import files
archive = pathlib.Path('/content/phase4_followup_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and 'isolated_index' not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
print(f'Results: {archive.stat().st_size / 1e6:.1f} MB')
files.download(str(archive))